# Localización temporal de la Magnetic Pileup Boundary de Marte con MAVEN MAG y SWEA

**Trabajo final: Machine Learning en Geofluidos — Modelo V7b (CNN multimodal con ranking).**  
**Estado:** cuaderno reproducible; las tablas y figuras observacionales se calculan al ejecutarlo a partir de datos reales, nunca de valores sintéticos presentados como observaciones.

### Objetivos

1. Introducir la interacción viento solar–Marte y la *magnetic pileup boundary* (MPB).
2. Documentar y ejecutar la preparación de ventanas MAG+SWEA con cadencias propias, controles de calidad, etiquetas débiles y particiones temporales.
3. Entrenar o recuperar V7b con los hiperparámetros exploratorios **\(\lambda_{\rm rank}=2\), margen \(m=0.25\), \(\lambda_{\rm region}=0.5\)** y reconstrucción auxiliar \(\lambda_{\rm rec}=0.15\).
4. Medir errores de localización en los cuatro folds; examinar curvas de entrenamiento y candidatos temporales.
5. Explorar los mapas latentes de ambos instrumentos, su organización en componentes principales y sus limitaciones físicas.

**Ejecución independiente:** el cuaderno detecta si tiene disponible el código V7b; de lo contrario, obtiene automáticamente una revisión fija desde GitHub y verifica las dependencias. No requiere ejecutar scripts manualmente antes. **Sí requiere acceso a observaciones reales:** caché MAG/SWEA existente o credenciales autorizadas de Google BigQuery. El entrenamiento completo consume tiempo de CPU y puede generar cargos BigQuery; por seguridad solo comienza al activar `RUN_TRAINING = True`.

**Limitación científica:** la evaluación encuentra el pico en un intervalo de ±30 min **alrededor de una MPB ya catalogada**; no equivale a detectar MPB desconocidas en órbitas completas.


## 1. Contexto físico

Marte no presenta un dipolo magnético global dominante como el terrestre, aunque posee campos remanentes de origen cortical. El viento solar transporta el campo magnético interplanetario; al interactuar con el planeta, se forma una región de plasma desacelerado detrás del **bow shock (BS)**, denominada **magnetovaina**. Más cerca del planeta aparece la **región de acumulación magnética (MPR)**, perteneciente a la magnetosfera inducida. La **MPB** separa aproximadamente magnetovaina y MPR: sus firmas pueden incluir cambios de intensidad y fluctuaciones del campo magnético y cambios espectrales en los electrones supratérmicos. La respuesta depende de condiciones del viento solar, geometría y campos corticales.

La misión **MAVEN** proporciona observaciones complementarias: **MAG** mide \(B_x,B_y,B_z\) y **SWEA** mide distribuciones de electrones, que en este trabajo se reducen a una serie espectral promedio por tiempo. Combinarlas permite estudiar una transición entre regímenes de plasma, no solo una discontinuidad en \(|\mathbf B|\).

**Figura 1.** Esquema conceptual elaborado aquí (no a escala ni una trayectoria MAVEN medida). Para un diagrama publicado, ver [LASP/MOP: Mars–solar-wind interaction](https://lasp.colorado.edu/mop/files/2012/04/14Mars.jpg).

**Bibliografía introductoria:** [MAVEN Magnetic Field Investigation (Connerney et al., 2015)](https://doi.org/10.1007/s11214-015-0169-4); [Martian bow shock and MPB models (2020)](https://doi.org/10.1029/2020JA028509); [Martian MPB current layer (2020)](https://doi.org/10.1029/2020GL089230).


In [ ]:
# 1.1 Entorno portátil: se ejecuta desde una carpeta cualquiera (local o Colab).
import os, sys, subprocess, importlib.util, json, hashlib, warnings
from pathlib import Path

REPO_URL = 'https://github.com/CamilaMastronardi/MAVEN-MPB-DeepLearning.git'
# Commit con la separación src/preprocessing / src/model y los 3 barridos V7b.
SOURCE_COMMIT = '06f428d79ca9e2b3c6abf48336c9c29d70cfee4e'
OVERRIDE_ROOT = os.environ.get('MAVEN_V7B_REPO_DIR', '').strip()
START_DIRECTORY = Path.cwd().resolve()

if OVERRIDE_ROOT:
    PROJECT_ROOT = Path(OVERRIDE_ROOT).expanduser().resolve()
    if not (PROJECT_ROOT / 'src/model/v7b.py').is_file():
        raise FileNotFoundError('MAVEN_V7B_REPO_DIR debe apuntar al árbol V7b reorganizado.')
else:
    candidates = [START_DIRECTORY, START_DIRECTORY.parent]
    available = [p for p in candidates if (p / 'src/model/v7b.py').is_file()]
    if available:
        PROJECT_ROOT = available[0]
    else:
        PROJECT_ROOT = START_DIRECTORY / '_maven_v7b_source'
        if not (PROJECT_ROOT / '.git').is_dir():
            subprocess.run(['git','clone','--quiet','--branch','v7b-clean-restructure',
                            '--single-branch',REPO_URL,str(PROJECT_ROOT)],check=True)
        subprocess.run(['git','-C',str(PROJECT_ROOT),'fetch','--quiet','origin',
                        'v7b-clean-restructure'],check=True)
        subprocess.run(['git','-C',str(PROJECT_ROOT),'checkout','--quiet','--detach',SOURCE_COMMIT],check=True)

# Instala solamente dependencias faltantes, sin tocar las bibliotecas instaladas.
deps = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib',
        'scipy':'scipy', 'sklearn':'scikit-learn', 'torch':'torch',
        'google.cloud.bigquery':'google-cloud-bigquery'}
def module_available(name):
    try:
        return importlib.util.find_spec(name) is not None
    except ModuleNotFoundError:
        return False
missing = [package for module,package in deps.items() if not module_available(module)]
if missing:
    subprocess.run([sys.executable,'-m','pip','install','--quiet',*missing],check=True)

# Fuente y datos pueden vivir en carpetas distintas. Esto permite reutilizar
# las cachés y checkpoints del repositorio local antiguo sin duplicar BigQuery.
data_root_override = os.environ.get('MAVEN_V7B_DATA_ROOT','').strip()
if data_root_override:
    DATA_ROOT = Path(data_root_override).expanduser().resolve()
else:
    data_candidates=[START_DIRECTORY,START_DIRECTORY.parent,PROJECT_ROOT]
    DATA_ROOT = next((d for d in data_candidates
                      if (d/'data/splits/cv4_monthly_manifest.csv').is_file()),PROJECT_ROOT)
if not (DATA_ROOT/'data/splits/cv4_monthly_manifest.csv').is_file():
    raise FileNotFoundError('No se encontró el manifiesto CV en DATA_ROOT.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print('Código V7b:', PROJECT_ROOT)
print('Datos y resultados locales:', DATA_ROOT)
print('Referencia:', SOURCE_COMMIT if not OVERRIDE_ROOT else 'carpeta local indicada por la usuaria')


In [ ]:
# 1.2 Configuración central: cambiar aquí, no en celdas dispersas.
from dataclasses import dataclass
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from IPython.display import display

@dataclass(frozen=True)
class Config:
    rank_weight: float = 2.0
    rank_margin: float = 0.25
    region_weight: float = 0.5  # Provisional mientras se analiza el tercer barrido.
    reconstruction_weight: float = 0.15
    learning_rate: float = 3e-4
    batch_size: int = 16
    seed: int = 42
    max_epochs: int = 100
    min_epochs: int = 10
    patience: int = 8
    rank_control_step_s: int = 120
    rank_positives_per_event: int = 2
    rank_hard_negatives: int = 2
    rank_random_negatives: int = 2
    evaluation_stride_s: int = 32
    n_folds: int = 4

CFG = Config()
RUN_TRAINING = False  # Poner True SOLO para entrenar/consultar BigQuery.
PILOT_EVENTS_PER_FOLD = 0  # 0 = catálogo completo; 8 = piloto por fold.
FOLD_TO_VISUALIZE = 1
MAX_LATENT_EVENTS = 24
TORCH_THREADS = 2

OUT_DIR = Path(os.environ.get('MAVEN_V7B_NOTEBOOK_RESULTS',
                              str(DATA_ROOT/'results/metrics/v7b_notebook_rank2_margin0p25_region0p5')))
FIG_DIR = Path(os.environ.get('MAVEN_V7B_NOTEBOOK_FIGURES',
                              str(DATA_ROOT/'results/figures/v7b_notebook')))
FIG_DIR.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(TORCH_THREADS)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
print('Parámetros del experimento:', CFG)
print('Salida de entrenamiento:', OUT_DIR)


In [ ]:
# Figura 1: esquema físico original y reproducible (sin representar geometría real exacta).
from matplotlib.patches import Circle, FancyArrowPatch
fig, ax = plt.subplots(figsize=(10,5.5))
ax.set_aspect('equal')
planet = Circle((0,0), .92, facecolor='#c27b55', edgecolor='#7c4629', lw=2)
ax.add_patch(planet)
ys=np.linspace(-2.65,2.65,400)
for x0,k,color,label in [(1.35,.20,'#2979a6','MPB'), (2.45,.27,'#4f9a70','Bow shock')]:
    x = x0-k*ys**2
    ax.plot(x,ys,color=color,lw=3,label=label)
for y in [-1.8,-1.2,-.6,0,.6,1.2,1.8]:
    ax.add_patch(FancyArrowPatch((4.2,y),(2.9,y),arrowstyle='-|>',
                                 mutation_scale=14,color='#d68b32',lw=1.6))
ax.annotate('Viento solar',xy=(3.1,2.35),xytext=(3.1,2.35),ha='center',fontsize=11)
ax.text(0,0,'MARTE',color='white',weight='bold',ha='center',va='center',fontsize=12)
ax.text(1.85,-2.4,'Magnetovaina',ha='center',fontsize=11)
ax.text(.5,-1.5,'MPR',ha='center',fontsize=11)
ax.set_xlim(-2.9,4.5);ax.set_ylim(-3,3)
ax.axis('off');ax.legend(loc='upper left',frameon=False)
ax.set_title('Interacción viento solar–Marte: esquema conceptual (no a escala)')
fig.tight_layout(); fig.savefig(FIG_DIR/'01_esquema_fisico.png',dpi=180,bbox_inches='tight')
plt.show()


## 2. Catálogo, instrumentos y particiones

El catálogo contiene **569 cruces MPB**; el manifiesto registra cobertura temporal MAG para 569 y SWEA para 559. La cantidad efectiva de ventanas conjuntas válidas depende de los datos medidos y sus gaps: **no** inferimos señales SWEA donde faltan observaciones.

| Instrumento | Entrada de V7b | Cadencia aproximada | Transformación |
|---|---|---|---|
| MAG | \(B_x,B_y,B_z,|\mathbf B|\) (4) | 1 s | normalización por canal, solo FIT |
| SWEA | 12 energías logarítmicas entre 30–300 eV | ~32 s | promedio angular, interpolación **en energía**, `log1p`, normalización solo FIT |

Cada candidato tiene un soporte temporal de **600 s**, independientemente de su cadencia nativa. Se rechazan ventanas incompletas, con gaps incompatibles o valores inválidos. No se interpola SWEA en el tiempo.

La validación usa **cuatro folds por mes UTC**, con separación adicional de meses FIT/MONITOR dentro de TRAIN. Se excluyen soportes temporales que se solapen. **El monitor y el conjunto externo nunca aportan muestras al cálculo de la normalización FIT.**


In [ ]:
# 2.1 Estadísticas del catálogo y asignación mensual; no usa BigQuery.
from src.preprocessing.catalog import read_manifest
from src.preprocessing.cv_splits import split_inner_monitor_months, check_disjoint_supports
from src.preprocessing.swea_cache import ENERGY_EV, SweaEventCache
from src.preprocessing.mag_cache import MagEventCache
from src.preprocessing.native_windows import extract_clip, collate_dual
from src.preprocessing import bigquery_source as bq

MANIFEST = DATA_ROOT/'data/splits/cv4_monthly_manifest.csv'
rows = read_manifest(MANIFEST)
for r in rows:
    r['bs_time'] = bq.to_utc(r['bs_timestamp_utc'])
print('Cruces anotados:', len(rows),
      '| MAG en manifiesto:',sum(r['mag_available']=='1' for r in rows),
      '| SWEA en manifiesto:',sum(r['swea_available']=='1' for r in rows))

counts = pd.DataFrame([{'fold':r['fold'],'grupo':r['source_group'],
                        'conjunto':r['mag_available']=='1' and r['swea_available']=='1'} for r in rows])
fig,ax=plt.subplots(figsize=(8,4))
ct=pd.crosstab(counts['fold'],counts['grupo'])
ct.plot(kind='bar',stacked=True,ax=ax)
ax.set(xlabel='Fold externo',ylabel='Número de cruces',title='Distribución del catálogo por fold y origen')
ax.legend(title='Catálogo');fig.tight_layout()
fig.savefig(FIG_DIR/'02_catalogo_por_fold.png',dpi=180);plt.show()

for k in range(1,5):
    train=[r for r in rows if r['fold']!=k and r['mag_available']=='1' and r['swea_available']=='1']
    val=[r for r in rows if r['fold']==k and r['mag_available']=='1' and r['swea_available']=='1']
    check_disjoint_supports(train,val)
    fit,mon,excluded=split_inner_monitor_months(train,k,CFG.seed)
    print(f'Fold {k}: FIT {len(fit)}, MONITOR {len(mon)}, VAL {len(val)}, FIT excluidos por solapamiento {excluded}')


### 2.2 Preparación de una observación real

La siguiente celda muestra un evento del fold elegido **solo si sus archivos de MAG y SWEA están disponibles en la caché verificada**. Si no están presentes, no se generan curvas ficticias. Al activar el entrenamiento, el mismo código V7b obtiene los bloques faltantes de BigQuery tras un *dry-run* y los guarda en la caché. Para una ejecución en Colab con consultas nuevas, la autenticación debe estar configurada con permisos en el proyecto BigQuery.


In [ ]:
# 2.3 Acceso seguro a las mismas cachés usadas por el entrenamiento.
import importlib.util
from src.model.v7b import MagSweaRegionAutoencoder, masked_region_pool
TRAIN_PATH = PROJECT_ROOT/'scripts/02_training/01_train_v7b.py'
spec = importlib.util.spec_from_file_location('maven_v7b_trainer',TRAIN_PATH)
trainer = importlib.util.module_from_spec(spec)
spec.loader.exec_module(trainer)

SQL = trainer.query_sources()
cache_mag=MagEventCache(DATA_ROOT/'data/cache/mag_events',project=bq.PROJECT,
                        table=bq.MAG,sql=SQL['MAG'])
cache_swea=SweaEventCache(DATA_ROOT/'data/cache/swea_events',project=bq.PROJECT,
                           table=bq.SWEA+' + '+bq.AXES,sql=SQL['SWEA'])

def load_cached_pair(r):
    start,end=trainer.event_bounds(r)
    m=cache_mag.load(r['time'],start,end,max_rows=25000)
    s=cache_swea.load(r['time'],start,end,max_rows=2000)
    if m is None or s is None or not len(m[0]) or not len(s[0]):
        return None
    return {'MAG':m,'SWEA':s}

def find_cached_event(fold, limit=None):
    eligible=[r for r in rows if r['fold']==fold and r['swea_available']=='1' and r['mag_available']=='1']
    for r in eligible[:limit]:
        pair=load_cached_pair(r)
        if pair is not None and extract_clip(pair['MAG'],pair['SWEA'],r['time'].timestamp()) is not None:
            return r,pair
    return None,None

def plot_cached_observation(row, pair):
    if row is None or pair is None:
        print('Sin caché MAG/SWEA conjunta utilizable: no se simulan mediciones.')
        return
    t0=row['time'].timestamp()
    mt,mx=pair['MAG']; st,sx=pair['SWEA']
    fig,(ax1,ax2)=plt.subplots(2,1,figsize=(11,7),sharex=True,layout='constrained')
    for j,label in enumerate(['Bx','By','Bz','|B|']):
        ax1.plot((mt-t0)/60,mx[:,j],lw=.8,label=label)
    ax1.set(ylabel='Campo magnético (nT)',title=f'MAVEN: cruce catalogado {row["time"].isoformat()}')
    ax1.legend(ncol=4,loc='upper right',fontsize=9)
    im=ax2.pcolormesh((st-t0)/60,ENERGY_EV,sx.T,shading='nearest',cmap='viridis')
    ax2.set(ylabel='Energía SWEA (eV)',xlabel='Minutos respecto a la MPB catalogada',yscale='log')
    fig.colorbar(im,ax=ax2,label='log(1 + flujo medio angular)')
    for ax in (ax1,ax2): ax.axvline(0,c='crimson',ls='--',lw=1.4)
    fig.savefig(FIG_DIR/'03_MAG_SWEA_evento_real.png',dpi=170)
    plt.show()

sample_row,sample_pair=find_cached_event(FOLD_TO_VISUALIZE)
plot_cached_observation(sample_row,sample_pair)


## 3. Construcción de candidatos y etiquetas débiles

- **Positivos:** centros en \(t_{\rm MPB}-30\), \(t_{\rm MPB}\) y \(t_{\rm MPB}+30\) segundos, siempre que la asociación MPB–BS sea admisible y se cumplan las reglas de exclusión.
- **Controles negativos débiles:** candidatos desplazados entre aproximadamente **10 y 30 min**, con paso de 120 s. Se excluyen intervalos demasiado próximos a MPB/BS conocidos, ventanas con cobertura deficiente o controles incompatibles con el orden de regiones.
- **Orientación:** un cruce de entrada BS→MPB tiene magnetovaina antes y región inducida después; el cruce de salida invierte ese orden. **Estas son etiquetas aproximadas derivadas del catálogo**, no una validación directa del estado plasmático.
- **Lotes de ranking:** por evento hasta dos positivos, dos negativos difíciles de épocas previas y dos controles aleatorios nuevos. Se comparan candidatos *del mismo cruce*.

La red recibe solamente las señales físicas normalizadas y los tiempos **relativos al candidato**, nunca las fechas de etiqueta, el índice de fold ni el grupo del catálogo.


In [ ]:
# Figura 4: posiciones de los centros de candidatos (esquema del muestreo).
from src.preprocessing.weak_regions import candidate_ranking_grid
fig,ax=plt.subplots(figsize=(10,2.8))
ax.axvspan(-35,35,color='#ffbbb7',alpha=.6,label='Positivos: ±35 s (regla física)')
for x in [-30,0,30]: ax.plot(x,1,'o',color='#b21e35',ms=9)
negative_offsets=np.r_[np.arange(-1800,-599,120),np.arange(600,1801,120)]
ax.scatter(negative_offsets,np.zeros(len(negative_offsets)),s=24,color='#4577ac',label='Centros de controles posibles')
ax.axvspan(-600,600,color='#dddddd',alpha=.28,label='Banda ambigua (aprox.)')
ax.axvline(0,ls='--',color='black',lw=1)
ax.set(xlim=(-1870,1870),ylim=(-.35,1.4),yticks=[],xlabel='Segundos respecto de MPB catalogada',
       title='Muestreo temporal de candidatos V7b: las reglas de exclusión filtran los controles')
ax.legend(loc='upper center',ncol=3,fontsize=8)
fig.tight_layout();fig.savefig(FIG_DIR/'04_candidatos_debiles.png',dpi=180);plt.show()


## 4. Arquitectura MAG+SWEA y función objetivo

Cada sensor pasa por un **encoder convolucional residual** y un **decoder auxiliar**. MAG tiene 4 canales y cuatro bloques residuales de dilataciones 1, 2, 4 y 8; SWEA tiene 12 canales y dos bloques de dilataciones 1, 2. Ambos producen **16 canales latentes temporales** (no un solo vector de 16 coordenadas por evento). Los mapas se resumen a la izquierda (< −60 s) y a la derecha (> +60 s), con máscaras que respetan la cadencia original. La fusión genera logits de región para ambos lados y un logit de transición para el centro candidato.

La función de pérdida es

$$
\mathcal L=\mathcal L_{\mathrm{BCE,trans}}+
\lambda_{\rm reg}\mathcal L_{\mathrm{BCE,reg}}+
\lambda_{\rm rec}(\mathcal L_{\rm MSE,MAG}+\mathcal L_{\rm MSE,SWEA})+
\lambda_{\rm rank}\mathcal L_{\rm rank},
$$

con

$$
\mathcal L_{\rm rank} = \left\langle
\operatorname{softplus}\left(m+z_{\rm neg}-z_{\rm pos}\right)
\right\rangle_{\text{pares dentro del evento}}.
$$

Los \(z\) son **logits**, no probabilidades. El componente de reconstrucción es **auxiliar**: no significa que el modelo sea un autoencoder no supervisado ni que se compriman todos los tiempos a una única coordenada. El monitor interno selecciona el checkpoint usando BCE de transición + peso de BCE de regiones + peso de ranking, sin incluir MSE en ese monitor.


In [ ]:
# Figura 5: arquitectura esquematizada a partir del modelo realmente importado.
from matplotlib.patches import FancyBboxPatch
net=MagSweaRegionAutoencoder()
print('Número de parámetros entrenables:',sum(p.numel() for p in net.parameters()))
fig,ax=plt.subplots(figsize=(12,5.2))
ax.set_xlim(0,12);ax.set_ylim(0,6);ax.axis('off')

def node(x,y,w,h,label,c):
    r=FancyBboxPatch((x,y),w,h,boxstyle='round,pad=0.06,rounding_size=0.12',
                     facecolor=c,edgecolor='#333333',lw=1.2)
    ax.add_patch(r);ax.text(x+w/2,y+h/2,label,ha='center',va='center',fontsize=9)
def arrow(x1,y1,x2,y2):
    ax.annotate('',xy=(x2,y2),xytext=(x1,y1),arrowprops={'arrowstyle':'->','lw':1.4})
node(.2,4.1,1.5,1,'MAG\n4 canales','#d3e7ff')
node(.2,1,1.5,1,'SWEA\n12 energías','#ffe0d1')
node(2.3,4.1,2.1,1,'Encoder MAG\n4 residuales / 16 z(t)','#c3d9fa')
node(2.3,1,2.1,1,'Encoder SWEA\n2 residuales / 16 z(t)','#facdb2')
node(5.1,2.55,2,1.1,'Pooling izq/der\nMAG + SWEA','#e3e1f5')
node(7.9,4.1,2,1,'Head regiones\n2 logits','#d0efd9')
node(7.9,1,2,1,'Head transición\n1 logit','#d9efd8')
node(10.45,2.65,1.25,.9,'Ranking\npor evento','#fff3bd')
for a,b in [((1.7,4.6),(2.3,4.6)),((1.7,1.5),(2.3,1.5)),
            ((4.4,4.6),(5.1,3.25)),((4.4,1.5),(5.1,2.9)),
            ((7.1,3.2),(7.9,4.5)),((7.1,2.8),(7.9,1.5)),((9.9,1.5),(10.45,2.8))]:arrow(*a,*b)
ax.text(3.2,5.35,'Decoder MAG → reconstrucción auxiliar',fontsize=8,ha='center')
ax.text(3.2,.45,'Decoder SWEA → reconstrucción auxiliar',fontsize=8,ha='center')
ax.set_title('Arquitectura V7b: representación multimodal, dos regiones y ranking')
fig.tight_layout();fig.savefig(FIG_DIR/'05_arquitectura_V7b.png',dpi=180);plt.show()


### 4.1 Elección exploratoria de hiperparámetros

Los coeficientes actuales provienen de barridos previos con los mismos cuatro folds. **Esta tabla documenta los resultados comunicados de esas corridas, no es una estimación nueva ni una evaluación ciega.**

**Barrido del peso de ranking:** la selección por monitor interno eligió 2, 2, 1 y 2 en los folds 1–4; la combinación por fold alcanzó 42.40 % dentro de 60 s, 64.76 % dentro de 120 s y una mediana de 72 s entre 559 eventos evaluables. Se fijó provisionalmente \(\lambda_{\rm rank}=2\) para el siguiente estudio.

**Barrido del margen, con \(\lambda_{\rm rank}=2\) fijo:**

| Margen | Eventos ≤60 s | Eventos ≤120 s | Mediana |
|:---:|---:|---:|---:|
| **0.25** | **42.58 %** | **65.65 %** | 72 s |
| 0.5 | 41.86 % | 64.04 % | 72 s |
| 1.0 | 41.32 % | 65.47 % | 72 s |
| 2.0 | 41.68 % | 65.12 % | 72 s |

Se usa margen **0.25** como candidato exploratorio. El peso de regiones se mantiene en **0.5** mientras se analiza su propio barrido. Las diferencias de ~1–2 puntos porcentuales no prueban una mejora robusta; los mismos folds externos se han consultado durante la exploración.


In [ ]:
# 4.2 Barrido de margen: figura con resultados guardados, sin inventar valores.
sweep_csv=DATA_ROOT/'results/metrics/v7b_ranking_margin_sweep/comparison_by_fold_and_margin.csv'
if sweep_csv.is_file():
    sweep=pd.read_csv(sweep_csv)
    overall=sweep.loc[sweep['fold'].astype(str)=='all_selected_folds'].copy()
    overall=overall.sort_values('rank_margin')
    if len(overall)==4:
        fig,ax=plt.subplots(figsize=(8,4.5))
        x=overall['rank_margin'].astype(float).to_numpy()
        for key,label in [('within_60s_over_all','≤60 s'),('within_120s_over_all','≤120 s')]:
            ax.plot(x,100*overall[key].astype(float),marker='o',lw=2,label=label)
        ax.set(xlabel='Margen del ranking (logits)',ylabel='Cruces dentro del umbral (%)',
               title='Sensibilidad temporal al margen, peso de ranking = 2')
        ax.set_xticks(x);ax.legend();ax.grid(alpha=.2)
        fig.tight_layout();fig.savefig(FIG_DIR/'05b_barrido_margen.png',dpi=180);plt.show()
    else:
        print('La tabla de barrido no contiene cuatro filas globales comparables.')
else:
    print('El CSV del barrido de margen no está disponible localmente; se conserva solo la tabla documentada.')


## 5. Entrenamiento: configuración y ejecución reproducible

**Configuración exploratoria actual:** ranking 2; margen 0.25; regiones 0.5; reconstrucción 0.15; AdamW (\(\mathrm{lr}=3\times10^{-4}\), `weight_decay=1e-3`), batch 16, seed 42, máximo 100 épocas, mínimo 10, paciencia 8, control negativos cada 120 s, 2 positivos + 2 negativos difíciles + 2 aleatorios y evaluación con paso de 32 s.

El peso de regiones **0.5 es provisional** mientras se termina el barrido de ese coeficiente; actualizar esta celda/configuración si el nuevo análisis lo justifica. El entrenamiento y el monitor están dentro de cada fold externo. Los checkpoints incluyen la firma de los datos, particiones e hiperparámetros; se usa `--resume` sin mezclar configuraciones.

La celda siguiente puede entrenar desde cero **sin preparar scripts manuales**. Por seguridad, por defecto `RUN_TRAINING=False`: si hay checkpoints compatibles, el resto del cuaderno los analiza sin reentrenar. Si no existen, poner `RUN_TRAINING=True`; la descarga de datos faltantes desde BigQuery exige credenciales y está protegida por límites. Para obtener un piloto, poner `PILOT_EVENTS_PER_FOLD=8`, dejando claro que sus métricas no son la validación completa.


In [ ]:
# 5.1 Autenticación opcional y entrenamiento REAL (sin datos inventados).
# Este comando usa exactamente las mismas funciones de preprocessing, CNN y CV del proyecto.
def check_cloud_auth():
    try:
        import google.auth
        google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
        return
    except Exception:
        try:
            from google.colab import auth
            print('Solicitando autorización Colab para BigQuery...')
            auth.authenticate_user()
        except ImportError as exc:
            raise RuntimeError('No hay credenciales ADC. Configurar gcloud auth application-default login.') from exc

training_cmd = [sys.executable,'-u',str(TRAIN_PATH),
                '--folds','all','--pilot-events-per-fold',str(PILOT_EVENTS_PER_FOLD),
                '--manifest',str(MANIFEST),
                '--output-dir',str(OUT_DIR),
                '--mag-cache-dir',str(DATA_ROOT/'data/cache/mag_events'),
                '--swea-cache-dir',str(DATA_ROOT/'data/cache/swea_events'),
                '--rank-weight',str(CFG.rank_weight), '--rank-margin',str(CFG.rank_margin),
                '--region-weight',str(CFG.region_weight),
                '--reconstruction-weight',str(CFG.reconstruction_weight),
                '--learning-rate',str(CFG.learning_rate),
                '--batch-size',str(CFG.batch_size), '--seed',str(CFG.seed),
                '--max-epochs',str(CFG.max_epochs), '--min-epochs',str(CFG.min_epochs),
                '--patience',str(CFG.patience),
                '--rank-control-step-s',str(CFG.rank_control_step_s),
                '--rank-positives-per-event',str(CFG.rank_positives_per_event),
                '--rank-hard-negatives',str(CFG.rank_hard_negatives),
                '--rank-random-negatives',str(CFG.rank_random_negatives),
                '--evaluation-stride-s',str(CFG.evaluation_stride_s),
                '--torch-threads',str(TORCH_THREADS), '--resume','--execute']
if RUN_TRAINING:
    # Si la corrida de referencia ya está terminada, evitar 4 entrenamientos idénticos.
    # Se comprueba la configuración antes de omitir, no solo la existencia del archivo.
    reference = DATA_ROOT/'results/metrics/v7b_ranking_margin_sweep/margin_0p25'
    ref_summary = reference/'cv_summary.json'
    reuse_existing = False
    if PILOT_EVENTS_PER_FOLD == 0 and ref_summary.is_file():
        meta = json.loads(ref_summary.read_text())
        opts = meta.get('training_configuration', {})
        expected = {'rank_weight': CFG.rank_weight, 'rank_margin': CFG.rank_margin,
                    'region_weight': CFG.region_weight,
                    'reconstruction_weight': CFG.reconstruction_weight,
                    'seed': CFG.seed, 'learning_rate': CFG.learning_rate,
                    'batch_size': CFG.batch_size, 'max_epochs': CFG.max_epochs,
                    'manifest_sha256': hashlib.sha256(MANIFEST.read_bytes()).hexdigest()}
        reuse_existing = (all(opts.get(k) == v for k,v in expected.items())
            and meta.get('pilot_subset') is False
            and sorted(meta.get('completed_folds', [])) == [1,2,3,4]
            and all((reference/f'fold_{k:02d}_final.pt').is_file()
                    and (reference/f'fold_{k:02d}_metrics.json').is_file()
                    and (reference/f'fold_{k:02d}_event_metrics.csv').is_file()
                    for k in range(1,5)))
    if reuse_existing:
        print('Ya existe una corrida completa compatible:',reference)
        print('Se reutilizará para figuras y PCA; no se reentrena.')
    else:
        # Las credenciales son necesarias si la cache no está completa; se comprueban
    # antes de lanzar cualquier extracción física potencialmente facturable.
        try:
            import google.colab
        except ImportError:
            pass  # WSL/local: los datos pueden estar completamente en caché.
        else:
            check_cloud_auth()
        OUT_DIR.mkdir(parents=True,exist_ok=True)
        subprocess.run(training_cmd,cwd=PROJECT_ROOT,check=True)
else:
    print('Entrenamiento desactivado; para ejecutarlo poner RUN_TRAINING=True.')
    print('Comando preparado:\n', ' '.join(training_cmd))


## 6. Métricas y diagnóstico del entrenamiento

Se cargan solamente resultados de una corrida **compatible con la configuración indicada**. No se confunden corridas piloto y completas. Si no existen checkpoints ni métricas, estas figuras se omiten y el cuaderno lo explica.

El error absoluto temporal es \(\epsilon_i=|\hat t_{\mathrm{MPB},i}-t_{\mathrm{MPB},i}|\). Reportamos mediana, percentil 90, fracciones de eventos con \(\epsilon\le60\) s y \(\epsilon\le120\) s, además del número de eventos realmente evaluables. Los scores de candidato y sus métricas de etiquetas débiles **no** constituyen precisión/recobrado de detección en órbita completa.


In [ ]:
# 6.1 Selección estricta de resultados existentes: no se mezclan hiperparámetros.
# Si el entrenamiento acaba de generar datos, ahora se puede dibujar el ejemplo MAG/SWEA.
if RUN_TRAINING and sample_pair is None:
    sample_row,sample_pair=find_cached_event(FOLD_TO_VISUALIZE)
    plot_cached_observation(sample_row,sample_pair)
import csv

def valid_results(folder):
    summary_path=folder/'cv_summary.json'
    if not summary_path.is_file():
        return False
    summary=json.loads(summary_path.read_text())
    conf=summary.get('training_configuration',{})
    expected={'rank_weight':CFG.rank_weight,'rank_margin':CFG.rank_margin,
              'region_weight':CFG.region_weight,'reconstruction_weight':CFG.reconstruction_weight,
              'seed':CFG.seed,'learning_rate':CFG.learning_rate,
              'batch_size':CFG.batch_size,'max_epochs':CFG.max_epochs,
              'evaluation_stride_s':CFG.evaluation_stride_s,
              'manifest_sha256':hashlib.sha256(MANIFEST.read_bytes()).hexdigest()}
    if any(conf.get(k)!=v for k,v in expected.items()):
        return False
    if summary.get('pilot_subset') != bool(PILOT_EVENTS_PER_FOLD):
        return False
    return all((folder/f'fold_{k:02d}_metrics.json').is_file()
               and (folder/f'fold_{k:02d}_final.pt').is_file()
               and (folder/f'fold_{k:02d}_event_metrics.csv').is_file()
               for k in range(1,5))

runs=[OUT_DIR,
      DATA_ROOT/'results/metrics/v7b_ranking_margin_sweep/margin_0p25',
      DATA_ROOT/'results/metrics/v7b_region_weight_sweep/region_0p5']
if os.environ.get('MAVEN_V7B_EXISTING_RESULTS'):
    runs.insert(0,Path(os.environ['MAVEN_V7B_EXISTING_RESULTS']).expanduser())
run_dir=next((p for p in runs if valid_results(p)),None)
if run_dir is None:
    print('No hay una CV completa compatible. Ejecutá la celda de entrenamiento o configurá MAVEN_V7B_EXISTING_RESULTS.')
else:
    print('Resultados compatibles:',run_dir)
    fold_metrics={k:json.loads((run_dir/f'fold_{k:02d}_metrics.json').read_text()) for k in range(1,5)}
    all_events=pd.concat([pd.read_csv(run_dir/f'fold_{k:02d}_event_metrics.csv').assign(fold=k)
                          for k in range(1,5)],ignore_index=True)
    all_events['absolute_error_s']=pd.to_numeric(all_events['absolute_error_s'],errors='coerce')
    errs=all_events['absolute_error_s'].dropna().to_numpy()
    print(f'Eventos totales={len(all_events)}, evaluables={len(errs)}; '
          f'mediana={np.median(errs):.1f} s; p90={np.percentile(errs,90):.1f} s; '
          f'≤60 s={np.mean(errs<=60):.2%}; ≤120 s={np.mean(errs<=120):.2%}')


In [ ]:
# 6.2 Curvas de pérdidas y monitor. Traza valores observados de los 4 folds.
if run_dir is not None:
    fig,axes=plt.subplots(2,2,figsize=(12,8),layout='constrained')
    for k,ax in enumerate(axes.flat,1):
        h=pd.DataFrame(fold_metrics[k]['train_history'])
        if h.empty:
            ax.text(.5,.5,'Sin historial',transform=ax.transAxes,ha='center')
            continue
        ax.plot(h['epoch'],h['train_det'],label='BCE transición TRAIN')
        ax.plot(h['epoch'],h['val_det'],label='BCE transición MONITOR')
        ax.plot(h['epoch'],h['val_select'],label='Pérdida selección MONITOR')
        ax.axvline(fold_metrics[k]['best_epoch'],color='k',ls=':',lw=1,label='Mejor época')
        ax.set(xlabel='Época',ylabel='Pérdida',title=f'Fold {k}')
        ax.legend(fontsize=8)
    fig.suptitle('Evolución real del entrenamiento V7b (FIT y MONITOR)')
    fig.savefig(FIG_DIR/'06_curvas_entrenamiento.png',dpi=170);plt.show()
else:
    print('Curvas no disponibles sin resultados compatibles.')


In [ ]:
# 6.3 Distribución de errores y métricas entre folds.
if run_dir is not None:
    fig,axes=plt.subplots(1,2,figsize=(12,4.5),layout='constrained')
    error_values=all_events['absolute_error_s'].dropna()
    axes[0].hist(error_values,bins=np.arange(0,1840,60),edgecolor='white')
    for mark,style,label in [(60,'--','60 s'),(120,'-.','120 s')]:
        axes[0].axvline(mark,ls=style,color='black',label=label)
    axes[0].set(xlabel='Error absoluto de localización (s)',ylabel='Número de eventos',
                title='Distribución global de errores')
    axes[0].legend()
    group=[all_events.loc[all_events.fold==k,'absolute_error_s'].dropna().to_numpy()
           for k in range(1,5)]
    axes[1].boxplot(group,showfliers=False)
    axes[1].set_xticks(range(1,5),[f'Fold {k}' for k in range(1,5)])
    axes[1].axhline(120,ls='--',c='gray',lw=1)
    axes[1].set(ylabel='Error absoluto (s)',title='Variabilidad entre folds (sin outliers graficados)')
    fig.savefig(FIG_DIR/'07_errores_localizacion.png',dpi=170);plt.show()
    fold_table=pd.DataFrame([{'fold':k,'N':m['val_total_events'],'N_evaluable':m['val_evaluable_events'],
                             'mediana_s':m['median_abs_peak_error_s'],'p90_s':m['p90_abs_peak_error_s'],
                             'dentro_60':m['fraction_within_60s'],
                             'dentro_120':m['fraction_within_120s']} for k,m in fold_metrics.items()])
    display(fold_table)
else:
    print('Distribución de errores no disponible sin resultados compatibles.')


## 7. Ejemplo de localización: curva de puntuación temporal

Para un evento del **fold externo**, evaluamos cada centro de ventana conjunto MAG+SWEA dentro de ±30 min de la MPB catalogada. El tiempo del máximo de la función sigmoide del logit de transición es el tiempo predicho. **La anotación solo define el intervalo de evaluación y permite medir el error; no entra en la red como predictor.**

El modelo y los normalizadores se leen del checkpoint real elegido por el monitor interno. Se utiliza únicamente la caché físicamente medida.


In [ ]:
# 7.1 Helpers para checkpoint y scoring del evento de validación.
@torch.no_grad()
def load_fold_model(fold):
    if run_dir is None:
        return None,None
    info=fold_metrics[fold]
    checkpoint=torch.load(run_dir/f'fold_{fold:02d}_final.pt',map_location='cpu',weights_only=False)
    assert checkpoint['signature']==info['signature'], 'Checkpoint y métricas incompatibles.'
    model=MagSweaRegionAutoencoder().eval()
    model.load_state_dict(checkpoint['model'])
    norm={k:{'mean':np.asarray(v['mean']),'scale':np.asarray(v['scale'])}
          for k,v in checkpoint['normalizers'].items()}
    return model,norm

all_mpb_s=np.array([r['time'].timestamp() for r in rows])
all_bs_s=np.array([r['bs_time'].timestamp() for r in rows])

@torch.no_grad()
def score_event(row,pair,model,norm):
    proposals,_=trainer.gather_candidates([row],{row['time']:pair},
                        all_mpb_s,all_bs_s,evaluate=True,stride_s=CFG.evaluation_stride_s)
    prepared=proposals.get(row['time'],[])
    offsets,scores=[],[]
    for begin in range(0,len(prepared),CFG.batch_size):
        group=prepared[begin:begin+CFG.batch_size]
        batch=collate_dual([(clip, c.transition if c.transition is not None else 0,
                             (c.left_region,c.right_region) if c.left_region is not None else (0,0))
                            for c,clip in group],norm,device='cpu')
        out=model(batch['mag'],batch['mag_mask'],batch['mag_time'],
                  batch['swea'],batch['swea_mask'],batch['swea_time'])
        prob=torch.sigmoid(out['transition_logit']).numpy()
        offsets.extend([(c.center_s-row['time'].timestamp()) for c,clip in group])
        scores.extend(prob.tolist())
    return np.array(offsets),np.array(scores)

if run_dir is not None:
    demo_model,demo_norm=load_fold_model(FOLD_TO_VISUALIZE)
    if sample_pair is None or sample_row['fold']!=FOLD_TO_VISUALIZE:
        sample_row,sample_pair=find_cached_event(FOLD_TO_VISUALIZE)
    if sample_pair is not None:
        dx,dy=score_event(sample_row,sample_pair,demo_model,demo_norm)
        if len(dx):
            j=np.argmax(dy)
            fig,ax=plt.subplots(figsize=(10,4))
            ax.plot(dx/60,dy,'o-',ms=3,label='Score V7b')
            ax.axvline(0,c='crimson',ls='--',label='MPB catalogada')
            ax.axvline(dx[j]/60,c='darkgreen',ls=':',label=f'Pico estimado (error={abs(dx[j]):.0f} s)')
            ax.set(xlabel='Minutos respecto a la anotación MPB',ylabel='P(transición | ventana)',
                   title=f'Localización condicional – fold externo {FOLD_TO_VISUALIZE}')
            ax.legend();fig.tight_layout();fig.savefig(FIG_DIR/'08_score_temporal.png',dpi=180);plt.show()
        else: print('La ventana de ejemplo no tiene candidatos evaluables.')
    else: print('No hay evento con mediciones cacheadas para el ejemplo.')
else: print('No hay checkpoint compatible para el ejemplo de score.')


## 8. Análisis del espacio latente

Este análisis explora **qué representaciones aprende cada rama** y si están relacionadas con diferencias entre ventanas que contienen una MPB y controles etiquetados débilmente. No equivale a una clasificación física validada de cruces.

1. Para cada ventana se extraen los mapas latentes MAG \(z_B\in\mathbb R^{16\times T_{\rm MAG}}\) y SWEA \(z_e\in\mathbb R^{16\times T_{\rm SWEA}}\) **antes de los decoders**.
2. Se resumen por separado en lado izquierdo (\(t<-60\) s) y derecho (\(t>60\) s); cada sensor produce un vector de **32 características** (16 por lado). Las medias respetan la máscara y el tiempo físico de cada muestra.
3. Para un único **fold externo** se representan con PCA 2D las ventanas de MPB débilmente positiva frente a controles conservadores. PCA se ajusta **solo para visualizar** las ventanas externas elegidas; no transforma el input de la red ni interviene en el entrenamiento. La comparación entre folds requeriría alineación: cada modelo tiene una base latente propia.
4. También se visualizan los **16 mapas temporales** de ambas ramas para un cruce. Estas figuras no implican que los canales latentes tengan una interpretación física única.

Se usan muestras reales y checkpoints externos: si falta caché no se fabrican puntos en PCA.


In [ ]:
# 8.1 Extracción de embeddings del fold externo; solo material ya cacheado.
from sklearn.decomposition import PCA

@torch.no_grad()
def latent_vector(model,norm,clip,side_offset=60.0):
    batch=collate_dual([(clip,1,(0,1))],norm,device='cpu')
    zm,_=model.mag(batch['mag'],batch['mag_mask'])
    ze,_=model.swea(batch['swea'],batch['swea_mask'])
    vecs={}
    for name,z in [('MAG',zm),('SWEA',ze)]:
        mask=batch[name.lower()+'_mask']
        rel=batch[name.lower()+'_time']
        left=masked_region_pool(z,mask,rel,side='left',gap_s=side_offset)
        right=masked_region_pool(z,mask,rel,side='right',gap_s=side_offset)
        vecs[name]=torch.cat([left,right],dim=1).squeeze(0).numpy()
    return vecs,{'MAG':(zm.squeeze(0).numpy(),batch['mag_time'].squeeze(0).numpy()),
                 'SWEA':(ze.squeeze(0).numpy(),batch['swea_time'].squeeze(0).numpy())}

latent_records=[]
latent_example=None
if run_dir is not None:
    latent_model,latent_norm=load_fold_model(FOLD_TO_VISUALIZE)
    candidates=[r for r in rows if r['fold']==FOLD_TO_VISUALIZE and r['mag_available']=='1'
                and r['swea_available']=='1']
    accepted=0
    for row in candidates:
        pair=load_cached_pair(row)
        if pair is None:
            continue
        prepared,_=trainer.gather_candidates([row],{row['time']:pair},
                         all_mpb_s,all_bs_s,ranking=True,
                         rank_control_step_s=CFG.rank_control_step_s)
        positives=[(c,clip) for c,clip in prepared.get(row['time'],[]) if c.transition==1]
        negatives=[(c,clip) for c,clip in prepared.get(row['time'],[]) if c.transition==0]
        if not positives or not negatives:
            continue
        # Un positivo cercano a la anotación + dos controles seguros por evento.
        positives=sorted(positives,key=lambda item:abs(item[0].center_s-row['time'].timestamp()))
        for c,clip in positives[:1]+negatives[:2]:
            vectors,maps=latent_vector(latent_model,latent_norm,clip)
            latent_records.append({'event':row['time'].isoformat(),'fold':FOLD_TO_VISUALIZE,
                                   'group':row['source_group'],'weak_label':c.transition,
                                   'MAG':vectors['MAG'],'SWEA':vectors['SWEA']})
            if c.transition==1 and latent_example is None:
                latent_example={'row':row,'maps':maps,'clip':clip}
        accepted+=1
        if accepted>=MAX_LATENT_EVENTS: break
    print(f'Embeddings: {len(latent_records)} ventanas de {accepted} eventos externos (fold {FOLD_TO_VISUALIZE}).')
else:
    print('Sin checkpoints compatibles: no se calculan embeddings.')


In [ ]:
# 8.2 PCA comparativa MAG / SWEA. Color = etiqueta DÉBIL del catálogo, no truth de plasma.
if len(latent_records)>=6 and len({r['weak_label'] for r in latent_records})==2:
    fig,axes=plt.subplots(1,2,figsize=(12,5),layout='constrained')
    for sensor,ax in zip(['MAG','SWEA'],axes):
        x=np.stack([r[sensor] for r in latent_records])
        labels=np.array([r['weak_label'] for r in latent_records])
        pca=PCA(n_components=2,svd_solver='full')
        xy=pca.fit_transform(x)
        explained=pca.explained_variance_ratio_
        for value,name,color in [(1,'Centro MPB (etiqueta débil)','#d1495b'),
                                 (0,'Control conservador','#007f8c')]:
            subset=labels==value
            ax.scatter(xy[subset,0],xy[subset,1],alpha=.7,label=name,color=color,s=34)
        ax.set(xlabel=f'PC1 ({explained[0]:.1%})',ylabel=f'PC2 ({explained[1]:.1%})',
               title=f'Latentes {sensor} – fold externo {FOLD_TO_VISUALIZE}')
        ax.legend(fontsize=8)
    fig.suptitle('PCA exploratoria por instrumento; proyección retrospectiva de las representaciones')
    fig.savefig(FIG_DIR/'09_PCA_latente_MAG_SWEA.png',dpi=180);plt.show()
else:
    print('PCA no disponible: faltan suficientes ventanas reales con ambas etiquetas.')


In [ ]:
# 8.3 Mapas latentes z(canal, tiempo): respetan los instantes nativos de cada instrumento.
if latent_example is not None:
    fig,axes=plt.subplots(2,1,figsize=(11,7),layout='constrained')
    for ax,sensor in zip(axes,['MAG','SWEA']):
        zz,t=latent_example['maps'][sensor]
        # pcolormesh ubica cada columna en SU tiempo nativo, sin remuestrear.
        pc=ax.pcolormesh(t,np.arange(zz.shape[0]),zz,shading='nearest',cmap='coolwarm')
        ax.axvline(0,c='black',ls='--',lw=1)
        ax.set(xlabel='Tiempo relativo al candidato (s)',ylabel='Canal latente',
               title=f'{sensor}: {zz.shape[0]} mapas temporales')
        fig.colorbar(pc,ax=ax,label='Activación latente (u.a.)')
    fig.suptitle('Representaciones latentes para una ventana MPB evaluada externamente')
    fig.savefig(FIG_DIR/'10_mapas_latentes_temporales.png',dpi=180);plt.show()
else:
    print('Mapas no disponibles hasta disponer de mediciones y checkpoint del fold.')


## 9. Discusión, límites y próximos pasos

**Qué puede concluirse:** si las curvas de monitor se estabilizan; cómo cambia la exactitud de localización entre meses/folds; si las representaciones latentes permiten separar parcialmente candidatos MPB y controles débiles; qué episodios presentan errores grandes. La PCA y las reconstrucciones son herramientas de diagnóstico y **no** evidencia autónoma de dos poblaciones físicas verificadas.

**Qué todavía no puede concluirse:** precisión y recall de detección MPB en órbitas completas sin etiquetas exhaustivas; que los controles de catálogo sean ausencia física garantizada de MPB; que la clasificación de región sea verdad física independiente; ni que las coordenadas PCA de distintos modelos por fold tengan ejes comparables.

**Observaciones metodológicas para el final:** (a) los parámetros ranking=2 y margen=0.25 surgieron de una exploración anterior con los mismos folds externos; por ello el rendimiento reportado es exploratorio; (b) se necesitan datos de prueba independientes o una optimización completamente anidada para reportar generalización sin sesgo de selección; (c) comparar MAG-only, SWEA-only y multimodal puede medir el aporte de cada instrumento; (d) contrastar representaciones con diagnósticos físicos adicionales y cruces independientes antes de asignarles significado.

**Figuras generadas reproduciblemente:** `results/figures/v7b_notebook/01_...10_...png`. Las figuras basadas en mediciones solo se guardan si existen datos reales. El archivo del notebook no incluye datos MAVEN, credenciales ni checkpoints.
